In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)

In [2]:
# ================================================================
# Imports
# ================================================================
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, coo_matrix, diags, bmat
from scipy.sparse.csgraph import connected_components
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
import geopandas as gpd
import pyreadr
import pickle
from IPython.display import clear_output


# ================================================================
# Load data (remove isolated points)
# ================================================================
no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

snow_cleaned_full = pyreadr.read_r("snow_cleaned_full.Rda")
snow_cleaned_full = list(snow_cleaned_full.values())[0]

all_y = snow_cleaned_full.drop(index=no_nbs).reset_index(drop=True)

coords_full = all_y.iloc[:, :2].to_numpy()
y_full      = all_y.iloc[:, 2:].to_numpy()

S_full, TT = y_full.shape
period = 52


# ================================================================
# Global time trend (scaled once)
# ================================================================
t_full = np.arange(1, TT + 1)
t_trend_full = (t_full - t_full.mean()) / t_full.std(ddof=0)


# ================================================================
# Build adjacency on FULL graph
# ================================================================
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords_full[:, 0], coords_full[:, 1]),
    crs="EPSG:4326"
)
gdf_aeqd = gdf.to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")
xy_full = np.vstack([gdf_aeqd.geometry.x, gdf_aeqd.geometry.y]).T / 1e6

Distances = squareform(pdist(xy_full))
A_full = (Distances <= 0.22).astype(int)
np.fill_diagonal(A_full, 0)
A_full = csr_matrix(A_full)

# ================================================================
# Connected components (FULL graph)
# ================================================================
n_comp, labels = connected_components(A_full, directed=False)
sizes = np.bincount(labels)

print("[INFO] component sizes:", sizes)

# ================================================================
# ★★★ MERGE THE TWO LARGEST COMPONENTS ★★★
# ================================================================
comp_order = np.argsort(sizes)[::-1]
comp1, comp2 = comp_order[:2]

print(
    f"[INFO] merging component {comp1} (size={sizes[comp1]}) "
    f"and component {comp2} (size={sizes[comp2]})"
)

keep = np.where(
    (labels == comp1) | (labels == comp2)
)[0]

print(f"[INFO] merged component size = {len(keep)}")

# ================================================================
# Subset data ONCE (merged component)
# ================================================================
coords = coords_full[keep]
xy     = xy_full[keep]
y      = y_full[keep]
S      = y.shape[0]


# ================================================================
# Rebuild adjacency (merged component)
# ================================================================
Distances = squareform(pdist(xy))
A = (Distances <= 0.22).astype(int)
np.fill_diagonal(A, 0)
A = csr_matrix(A)

deg = np.array(A.sum(axis=1)).flatten()
Q_icar = diags(deg) - A
I_S    = diags(np.ones(S))


# ================================================================
# Storage
# ================================================================
results = {}


# ================================================================
# Event loop (only p01, as in your current code)
# ================================================================
for event in ["p01"]:

    print(f"\n[EVENT] {event}")

    # ------------------------------------------------------------
    # Define p01
    # ------------------------------------------------------------
    loc_mask = (y[:, :-1] == 0)

    row_idx, time_idx = np.where(loc_mask)
    N = len(row_idx)

    next_y = y[row_idx, time_idx + 1]
    outcome = next_y
    kappa = outcome - 0.5

    print(f"    N = {N}")

    # ============================================================
    # Covariates
    # ============================================================
    t_raw   = time_idx + 1
    t_trend = t_trend_full[time_idx]

    cov = np.column_stack([
        np.ones(N), np.ones(N),
        np.cos(2*np.pi*t_raw / period),
        np.cos(2*np.pi*t_raw / period),
        np.sin(2*np.pi*t_raw / period),
        np.sin(2*np.pi*t_raw / period),
        t_trend, t_trend
    ])
    K = cov.shape[1]

    weeks_per_bin = 13
    time_bin = 52 // weeks_per_bin
    week_raw = (t_raw // weeks_per_bin) % time_bin


    # ============================================================
    # Design matrix X
    # ============================================================
    rows, cols, vals = [], [], []

    for i in tqdm(
        range(N),
        desc=f"Building X | merged | {event}",
        leave=False
    ):
        s = row_idx[i]
        w = week_raw[i]
        for k in range(K):
            col = s + S * (w + time_bin * k)
            rows.append(i)
            cols.append(col)
            vals.append(cov[i, k])

    X = coo_matrix(
        (vals, (rows, cols)),
        shape=(N, K * time_bin * S)
    ).tocsr()


    # ============================================================
    # Helpers
    # ============================================================
    def week_block_indices(w, K, period, S):
        idx = np.empty(K * S, dtype=int)
        p = 0
        for k in range(K):
            start = (k * period + w) * S
            idx[p:p+S] = np.arange(start, start + S)
            p += S
        return idx

    def theta_slice(k, w):
        start = (k * time_bin + w) * S
        return slice(start, start + S)


    # ============================================================
    # MCMC setup
    # ============================================================
    burn = 1000
    thin = 5
    tot_save = 1000
    total_iters = burn + tot_save * thin

    a0, b0 = 2, 1

    all_theta = np.zeros((K, time_bin, S, tot_save))
    all_tau   = np.zeros((K, time_bin, tot_save))
    save_idx = 0

    Q_template = []
    for k in range(K):
        Qk = Q_icar if (k % 2 == 0) else I_S
        for w in range(time_bin):
            Q_template.append(Qk)

    rhs = X.T @ kappa
    curr_theta = np.random.randn(K * S * time_bin)
    curr_tau   = np.ones(K * time_bin) * 1000


    # ============================================================
    # MCMC
    # ============================================================
    for it in tqdm(
        range(total_iters),
        desc=f"MCMC | merged | {event}",
        leave=True
    ):

        block_list = [
            curr_tau[j] * Q_template[j]
            for j in range(K * time_bin)
        ]

        curr_prec = bmat(
            [[block_list[i] if i == j else None
              for j in range(K * time_bin)]
             for i in range(K * time_bin)],
            format="csr"
        )

        phi = X @ curr_theta
        omega = random_polyagamma(1, phi)

        post_prec = X.T.multiply(omega) @ X + curr_prec

        for w in range(time_bin):
            idx = week_block_indices(w, K, time_bin, S)
            Qw = post_prec[np.ix_(idx, idx)]
            bw = rhs[idx]

            factor = cholesky(Qw)
            mu = factor.solve_A(bw)
            z  = factor.solve_A(np.random.randn(len(idx)))
            curr_theta[idx] = mu + z

        for k in range(K):
            for w in range(time_bin):
                j = k * time_bin + w
                sl = slice(j * S, (j + 1) * S)
                beta = curr_theta[sl]

                Qj = Q_icar if (k % 2 == 0) else I_S
                rk = (S - 1) if (k % 2 == 0) else S

                quad = beta @ (Qj @ beta)

                curr_tau[j] = np.random.gamma(
                    a0 + 0.5 * rk,
                    1.0 / (b0 + 0.5 * quad)
                )

        if it >= burn and (it - burn) % thin == 0:
            for k in range(K):
                for w in range(time_bin):
                    all_theta[k, w, :, save_idx] = curr_theta[theta_slice(k, w)]
            all_tau[:, :, save_idx] = curr_tau.reshape(K, time_bin)

            save_idx += 1

        if save_idx % 1 == 0:
            tau_mat = curr_tau.reshape(K, time_bin)
            df_tau = pd.DataFrame(
                tau_mat.T,
                index=[f"w{w}" for w in range(time_bin)],
                columns=[f"k{k}" for k in range(K)]
            )

            clear_output(wait=True)
            print(
                f"merged | event {event} | "
                f"iter {it} | save {save_idx}/{tot_save}"
            )
            display(df_tau)

        if save_idx == tot_save:
            break


    results[event] = {
        "keep_idx": keep,
        "theta": all_theta,
        "tau": all_tau
    }


# ================================================================
# Save everything
# ================================================================
with open("mcmc_merged_two_largest_components_p01.pkl", "wb") as f:
    pickle.dump(results, f)

print("\n[INFO] MERGED component p01 finished and saved.")


merged | event p01 | iter 5995 | save 1000/1000


,k0,k1,k2,k3,k4,k5,k6,k7
w0,735.387496,761.836183,597.707106,761.031071,0.913078,789.808875,603.395755,697.648747
w1,1.979281,772.941519,577.196487,829.698571,659.067596,751.636154,659.580829,739.846746
w2,1.497311,738.911783,688.401080,704.429529,661.669128,716.549944,545.314710,794.944112
w3,0.655022,740.226016,790.584018,762.588129,693.986205,803.871992,676.863575,753.468248


MCMC | merged | p01: 100%|█████████▉| 5995/6000 [2:32:58<00:07,  1.53s/it]



[INFO] MERGED component p01 finished and saved.


In [3]:
# ================================================================
# Imports
# ================================================================
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, coo_matrix, diags, bmat
from scipy.sparse.csgraph import connected_components
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
import geopandas as gpd
import pyreadr
import pickle
from IPython.display import clear_output


# ================================================================
# Load data (remove isolated points)
# ================================================================
no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

snow_cleaned_full = pyreadr.read_r("snow_cleaned_full.Rda")
snow_cleaned_full = list(snow_cleaned_full.values())[0]

all_y = snow_cleaned_full.drop(index=no_nbs).reset_index(drop=True)

coords_full = all_y.iloc[:, :2].to_numpy()
y_full      = all_y.iloc[:, 2:].to_numpy()

S_full, TT = y_full.shape
period = 52


# ================================================================
# Global time trend (scaled once)
# ================================================================
t_full = np.arange(1, TT + 1)
t_trend_full = (t_full - t_full.mean()) / t_full.std(ddof=0)


# ================================================================
# Build adjacency on FULL graph
# ================================================================
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords_full[:, 0], coords_full[:, 1]),
    crs="EPSG:4326"
)
gdf_aeqd = gdf.to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")
xy_full = np.vstack([gdf_aeqd.geometry.x, gdf_aeqd.geometry.y]).T / 1e6

Distances = squareform(pdist(xy_full))
A_full = (Distances <= 0.22).astype(int)
np.fill_diagonal(A_full, 0)
A_full = csr_matrix(A_full)

# ================================================================
# Connected components (FULL graph)
# ================================================================
n_comp, labels = connected_components(A_full, directed=False)
sizes = np.bincount(labels)

print("[INFO] component sizes:", sizes)

# ================================================================
# ★★★ MERGE THE TWO LARGEST COMPONENTS ★★★
# ================================================================
comp_order = np.argsort(sizes)[::-1]
comp1, comp2 = comp_order[:2]

print(
    f"[INFO] merging component {comp1} (size={sizes[comp1]}) "
    f"and component {comp2} (size={sizes[comp2]})"
)

keep = np.where(
    (labels == comp1) | (labels == comp2)
)[0]

print(f"[INFO] merged component size = {len(keep)}")

# ================================================================
# Subset data ONCE (merged component)
# ================================================================
coords = coords_full[keep]
xy     = xy_full[keep]
y      = y_full[keep]
S      = y.shape[0]


# ================================================================
# Rebuild adjacency (merged component)
# ================================================================
Distances = squareform(pdist(xy))
A = (Distances <= 0.22).astype(int)
np.fill_diagonal(A, 0)
A = csr_matrix(A)

deg = np.array(A.sum(axis=1)).flatten()
Q_icar = diags(deg) - A
I_S    = diags(np.ones(S))


# ================================================================
# Storage
# ================================================================
results = {}


# ================================================================
# Event loop (p10 ONLY)
# ================================================================
for event in ["p10"]:

    print(f"\n[EVENT] {event}")

    # ------------------------------------------------------------
    # Define p10 correctly: P(X_{t+1}=0 | X_t=1)
    # ------------------------------------------------------------
    loc_mask = (y[:, :-1] == 1)

    row_idx, time_idx = np.where(loc_mask)
    N = len(row_idx)

    next_y = y[row_idx, time_idx + 1]

    # p10 outcome: 1{X_{t+1}=0}
    outcome = 1 - next_y
    kappa = outcome - 0.5

    print(f"    N = {N}")

    # ============================================================
    # Covariates
    # ============================================================
    t_raw   = time_idx + 1
    t_trend = t_trend_full[time_idx]

    cov = np.column_stack([
        np.ones(N), np.ones(N),
        np.cos(2*np.pi*t_raw / period),
        np.cos(2*np.pi*t_raw / period),
        np.sin(2*np.pi*t_raw / period),
        np.sin(2*np.pi*t_raw / period),
        t_trend, t_trend
    ])
    K = cov.shape[1]

    weeks_per_bin = 26
    time_bin = 52 // weeks_per_bin
    week_raw = (t_raw // weeks_per_bin) % time_bin


    # ============================================================
    # Design matrix X
    # ============================================================
    rows, cols, vals = [], [], []

    for i in tqdm(
        range(N),
        desc=f"Building X | merged | {event}",
        leave=False
    ):
        s = row_idx[i]
        w = week_raw[i]
        for k in range(K):
            col = s + S * (w + time_bin * k)
            rows.append(i)
            cols.append(col)
            vals.append(cov[i, k])

    X = coo_matrix(
        (vals, (rows, cols)),
        shape=(N, K * time_bin * S)
    ).tocsr()


    # ============================================================
    # Helpers
    # ============================================================
    def week_block_indices(w, K, period, S):
        idx = np.empty(K * S, dtype=int)
        p = 0
        for k in range(K):
            start = (k * period + w) * S
            idx[p:p+S] = np.arange(start, start + S)
            p += S
        return idx

    def theta_slice(k, w):
        start = (k * time_bin + w) * S
        return slice(start, start + S)


    # ============================================================
    # MCMC setup
    # ============================================================
    burn = 1000
    thin = 5
    tot_save = 1000
    total_iters = burn + tot_save * thin

    a0, b0 = 2, 1

    all_theta = np.zeros((K, time_bin, S, tot_save))
    all_tau   = np.zeros((K, time_bin, tot_save))
    save_idx = 0

    Q_template = []
    for k in range(K):
        Qk = Q_icar if (k % 2 == 0) else I_S
        for w in range(time_bin):
            Q_template.append(Qk)

    rhs = X.T @ kappa
    curr_theta = np.random.randn(K * S * time_bin)
    curr_tau   = np.ones(K * time_bin) * 1000


    # ============================================================
    # MCMC
    # ============================================================
    for it in tqdm(
        range(total_iters),
        desc=f"MCMC | merged | {event}",
        leave=True
    ):

        block_list = [
            curr_tau[j] * Q_template[j]
            for j in range(K * time_bin)
        ]

        curr_prec = bmat(
            [[block_list[i] if i == j else None
              for j in range(K * time_bin)]
             for i in range(K * time_bin)],
            format="csr"
        )

        phi = X @ curr_theta
        omega = random_polyagamma(1, phi)

        post_prec = X.T.multiply(omega) @ X + curr_prec

        for w in range(time_bin):
            idx = week_block_indices(w, K, time_bin, S)
            Qw = post_prec[np.ix_(idx, idx)]
            bw = rhs[idx]

            try:
                factor = cholesky(Qw)
            except Exception as e:
                jitter = 1
                print(
                    f"[WARN] Cholesky failed at "
                    f"(event={event}, iter={it}, w={w}) — "
                    f"adding jitter={jitter}"
                )
                factor = cholesky(Qw + jitter * diags(np.ones(Qw.shape[0])))
            mu = factor.solve_A(bw)
            z  = factor.solve_A(np.random.randn(len(idx)))
            curr_theta[idx] = mu + z

        for k in range(K):
            for w in range(time_bin):
                j = k * time_bin + w
                sl = slice(j * S, (j + 1) * S)
                beta = curr_theta[sl]

                Qj = Q_icar if (k % 2 == 0) else I_S
                rk = (S - 1) if (k % 2 == 0) else S

                quad = beta @ (Qj @ beta)

                curr_tau[j] = np.random.gamma(
                    a0 + 0.5 * rk,
                    1.0 / (b0 + 0.5 * quad)
                )

        if it >= burn and (it - burn) % thin == 0:
            for k in range(K):
                for w in range(time_bin):
                    all_theta[k, w, :, save_idx] = curr_theta[theta_slice(k, w)]
            all_tau[:, :, save_idx] = curr_tau.reshape(K, time_bin)

            save_idx += 1

        if save_idx % 1 == 0:
            tau_mat = curr_tau.reshape(K, time_bin)
            df_tau = pd.DataFrame(
                tau_mat.T,
                index=[f"w{w}" for w in range(time_bin)],
                columns=[f"k{k}" for k in range(K)]
            )

            clear_output(wait=True)
            print(
                f"merged | event {event} | "
                f"iter {it} | save {save_idx}/{tot_save}"
            )
            display(df_tau)

        if save_idx == tot_save:
            break


    results[event] = {
        "keep_idx": keep,
        "theta": all_theta,
        "tau": all_tau
    }


# ================================================================
# Save everything
# ================================================================
with open("mcmc_merged_two_largest_components_p10.pkl", "wb") as f:
    pickle.dump(results, f)

print("\n[INFO] MERGED component p10 finished and saved.")


merged | event p10 | iter 5995 | save 1000/1000


,k0,k1,k2,k3,k4,k5,k6,k7
w0,2.193706,742.254913,3.713346,729.899939,750.511094,786.956135,641.870013,731.868001
w1,763.657525,762.449908,0.701770,758.894761,1.069318,744.354565,472.414402,702.831281


MCMC | merged | p10: 100%|█████████▉| 5995/6000 [1:05:38<00:03,  1.52it/s]



[INFO] MERGED component p10 finished and saved.
